In [2]:
import pandas as pd

# Load and inspect the tranco data 
- (flat file)

In [2]:
# Load the Tranco list — no header, just rank and domain
df = pd.read_csv('../raw_data/tranco_list.csv', header=None, names=['rank', 'domain'])
df.head(10)

,rank,domain
0,1,google.com
1,2,gtld-servers.net
2,3,cloudflare.com
3,4,gstatic.com
4,5,facebook.com
5,6,microsoft.com
6,7,googleapis.com
7,8,youtube.com
8,9,amazonaws.com
9,10,apple.com


In [3]:
# Check your column names
print(df.columns.tolist())

['rank', 'domain']


In [4]:
# Extracts the last "name" and "extension" before any slash
df[['site', 'extension']] = df['domain'].str.split('.', n=1, expand=True)
df.head(5)

,rank,domain,site,extension
0,1,google.com,google,com
1,2,gtld-servers.net,gtld-servers,net
2,3,cloudflare.com,cloudflare,com
3,4,gstatic.com,gstatic,com
4,5,facebook.com,facebook,com


In [5]:
df.extension.value_counts().head(10)

extension
com       423938
net        49080
org        37976
ru         37629
de         28962
co.uk      26103
com.br     14028
nl         13809
fr         11535
io         10224
Name: count, dtype: int64

- we see that there is some compound extension (com.XX // co.XX) so let's check if we have relevat data for the european domain to keep. 

In [6]:
# Extensions that CONTAIN 'com' anywhere (union of both above)
contains_com = df[df['extension'].str.contains('com.', na=False)]
print(f'Extensions containing com anywhere: {len(contains_com)} rows')
print(contains_com['extension'].value_counts().head(20))

Extensions containing com anywhere: 38673 rows
extension
com.br    14028
com.au     5518
com.ar     2376
com.tr     2271
com.ua     2162
com.cn     1807
com.tw     1436
com.mx     1420
com.co      958
com.my      606
com.vn      498
com.pl      476
com.pk      418
com.ph      370
com.sg      362
com.pe      358
com.hk      315
com.uy      294
com.bd      238
com.py      196
Name: count, dtype: int64


the goal is to use on and only simple identifiyer  by  working with the coumpond tld. 
Therefore, for a domain that hase the extension com.uk I only need and will be using the Uk part : the last sagment. 

In [7]:
def get_last_segment(ext):
    """Always returns the rightmost part — the true country indicator"""
    return str(ext).lower().strip().split('.')[-1]

In [8]:
df.shape

(1000000, 4)

In [9]:
df['last_segment'] = df['extension'].apply(get_last_segment)

df[['domain', 'extension', 'last_segment']].head(10)

,domain,extension,last_segment
0,google.com,com,com
1,gtld-servers.net,net,net
2,cloudflare.com,com,com
3,gstatic.com,com,com
4,facebook.com,com,com
5,microsoft.com,com,com
6,googleapis.com,com,com
7,youtube.com,com,com
8,amazonaws.com,com,com
9,apple.com,com,com


i just want to check if there are still any compound extension

In [10]:
df[df['extension'].isin(['co.uk', 'com.br', 'com', 'fr'])].groupby('extension').head(3)

,rank,domain,site,extension,last_segment
0,1,google.com,google,com,com
2,3,cloudflare.com,cloudflare,com,com
3,4,gstatic.com,gstatic,com,com
220,221,bbc.co.uk,bbc,co.uk,uk
312,313,amazon.co.uk,amazon,co.uk,uk
377,378,dailymail.co.uk,dailymail,co.uk,uk
496,497,gandi-ns.fr,gandi-ns,fr,fr
532,533,uol.com.br,uol,com.br,br
613,614,free.fr,free,fr,fr
625,626,amazon.fr,amazon,fr,fr


In [11]:
country_map = {
    # European
    'fr': 'FR', 
    'de': 'DE', 
    'uk': 'UK', 
    'nl': 'NL',
    'es': 'ES', 
    'it': 'IT', 
    'be': 'BE', 
    'ch': 'CH',
    'at': 'AT', 
    'pl': 'PL', 
    'se': 'SE', 
    'dk': 'DK',
    'fi': 'FI', 
    'pt': 'PT', 
    'no': 'NO',
    'ie': 'IE',
    'lu': 'LU', 
    'cz': 'CZ', 
    'ro': 'RO', 
    'hu': 'HU',
    'gr': 'GR', 
    'hr': 'HR', 
    'sk': 'SK', 
    'eu': 'EU',

    # Non-European (seen in our com.XX value_counts above)
    'br': 'BR', 
    'au': 'AU', 
    'ar': 'AR', 
    'tr': 'TR',
    'ua': 'UA',
    'cn': 'CN', 
    'tw': 'TW', 
    'mx': 'MX',
    'co': 'CO', 
    'my': 'MY', 
    'vn': 'VN', 
    'pk': 'PK',
    'ph': 'PH', 
    'sg': 'SG', 
    'pe': 'PE', 
    'hk': 'HK',
    'uy': 'UY', 
    'bd': 'BD', 
    'py': 'PY', 
    'ru': 'RU',
}

In [12]:
EUROPEAN = {
    'fr','de',
    'uk','nl',
    'es','it',
    'be','ch',
    'at','pl',
    'se','dk',
    'fi','pt',
    'no','ie',
    'lu','cz',
    'ro','hu',
    'gr','hr',
    'sk','eu'
    }
INTERNATIONAL = {'com','org','net','io','app','edu','gov','info','biz'}

In [13]:
def get_country(last_segment):
    return country_map.get(last_segment, 'Unknown')

In [14]:
def get_region(last_segment):
    if last_segment in EUROPEAN:      return 'European'
    if last_segment in INTERNATIONAL: return 'International'
    return 'Non-European'

In [15]:
def get_ext_type(ext):
    #i want to classify which of the 3 extension patterns this row is
    ext = str(ext).lower().strip()
    if ext.startswith('com.'): return 'com.XX'
    if ext.startswith('co.'):  return 'co.XX'
    return 'simple'

now let's create a mapping with the country , region  an the extention type

In [16]:
df['country']  = df['last_segment'].apply(get_country)
df.head(3)

,rank,domain,site,extension,last_segment,country
0,1,google.com,google,com,com,Unknown
1,2,gtld-servers.net,gtld-servers,net,net,Unknown
2,3,cloudflare.com,cloudflare,com,com,Unknown


In [17]:
df['region']   = df['last_segment'].apply(get_region)

In [18]:
df['ext_type'] = df['extension'].apply(get_ext_type)

In [19]:
df[['domain', 'extension', 'last_segment', 'ext_type', 'country', 'region']].head(10)

,domain,extension,last_segment,ext_type,country,region
0,google.com,com,com,simple,Unknown,International
1,gtld-servers.net,net,net,simple,Unknown,International
2,cloudflare.com,com,com,simple,Unknown,International
3,gstatic.com,com,com,simple,Unknown,International
4,facebook.com,com,com,simple,Unknown,International
5,microsoft.com,com,com,simple,Unknown,International
6,googleapis.com,com,com,simple,Unknown,International
7,youtube.com,com,com,simple,Unknown,International
8,amazonaws.com,com,com,simple,Unknown,International
9,apple.com,com,com,simple,Unknown,International


now let's check how these categories breackdown

In [20]:
df['ext_type'].value_counts()

ext_type
simple    919611
co.XX      41877
com.XX     38512
Name: count, dtype: int64

In [21]:
df.groupby('ext_type')['region'].value_counts()

ext_type  region       
co.XX     European          26154
          Non-European      15723
com.XX    Non-European      37776
          European            736
simple    International    537082
          Non-European     254733
          European         127796
Name: count, dtype: int64

In [22]:
df.groupby('region')['country'].value_counts()

region         country
European       UK          30902
               DE          28962
               NL          13809
               FR          11685
               IT           9913
               PL           8519
               ES           5588
               EU           5143
               CZ           4715
               SE           3716
               RO           3622
               CH           3486
               BE           3321
               GR           3193
               AT           2917
               HU           2670
               DK           2252
               PT           2202
               FI           2052
               SK           1760
               NO           1671
               IE           1286
               HR           1059
               LU            243
International  Unknown    537082
Non-European   Unknown    198798
               RU          37629
               BR          17586
               CN           9864
               AU   

In [23]:
df[(df['region'].isin(['European','International'])) & (df['country']=='Unknown')]['extension'].value_counts()

extension
com     423938
net      49080
org      37976
io       10224
info      5028
app       3677
edu       2999
biz       2426
gov       1734
Name: count, dtype: int64

now that we saw that we have a good amount

In [24]:
before = len(df)

df_clean = df[df['region'].isin(['European', 'International'])].copy()

In [25]:
df_clean.shape

(691768, 8)

In [26]:
df_clean['region'].value_counts()

region
International    537082
European         154686
Name: count, dtype: int64

In [27]:
(df_clean["region"]=="European").sum()

np.int64(154686)

In [28]:
(df_clean["region"]=="International").sum()

np.int64(537082)

In [29]:
before - len(df_clean)

308232

co.XX and com.XX extensions were all reduced to their segment to identify the true country. 

Non-European countries (BR, AU, AR, TR, CN etc.) were dropped as out of scope. 

Plain international TLDs (.com,.org, .net) were kept as a separate "International" category since major global platforms (Google, Meta, Amazon) are highly relevant trackers regardless of TLD.

### Let's sample down our working dataset
- top 350 European + top 150 international, ranked by popularity

In [30]:
countries = ['UK', 'DE', 'FR', 'IT']

df_eu = df_clean[
    (df_clean['region'] == 'European') &
    (df_clean['country'].isin(countries))
].sort_values('rank').head(3000)

df_intl = df_clean[
    (df_clean['region'] == 'International')
].sort_values('rank').head(2000)

In [31]:
df_work = pd.concat([df_eu, df_intl]).reset_index(drop=True)

In [32]:
# Add the full URL for the scraper, and blank columns to fill later
df_work['url']        = 'https://' + df_work['domain']
df_work['category']   = None   # to fill manually or via keyword lookup
df_work['risk_label']  = None  # to fill from scraper output + CNIL ground truth

In [33]:
df_work = df_work[['rank', 'url', 'domain', 'site', 'extension', 'last_segment',
                    'ext_type', 'country', 'region', 'category']]

In [34]:
print(f'Final working dataset: {len(df_work)} URLs')
print(df_work['region'].value_counts())
df_work.head(10)

Final working dataset: 5000 URLs
region
European         3000
International    2000
Name: count, dtype: int64


,rank,url,domain,site,extension,last_segment,ext_type,country,region,category
0,116,https://telekom.de,telekom.de,telekom,de,de,simple,DE,European,None
1,143,https://iiko.it,iiko.it,iiko,it,it,simple,IT,European,None
2,200,https://nominetdns.uk,nominetdns.uk,nominetdns,uk,uk,simple,UK,European,None
3,215,https://t-online.de,t-online.de,t-online,de,de,simple,DE,European,None
4,221,https://bbc.co.uk,bbc.co.uk,bbc,co.uk,uk,co.XX,UK,European,None
5,290,https://www.gov.uk,www.gov.uk,www,gov.uk,uk,simple,UK,European,None
6,313,https://amazon.co.uk,amazon.co.uk,amazon,co.uk,uk,co.XX,UK,European,None
7,359,https://rzone.de,rzone.de,rzone,de,de,simple,DE,European,None
8,378,https://dailymail.co.uk,dailymail.co.uk,dailymail,co.uk,uk,co.XX,UK,European,None
9,393,https://amazon.de,amazon.de,amazon,de,de,simple,DE,European,None


In [35]:
# Filter for International region
international_df = df_work[df_work["region"] == "International"]

# Display first 20 rows
display(international_df.head(20))

,rank,url,domain,site,extension,last_segment,ext_type,country,region,category
3000,1,https://google.com,google.com,google,com,com,simple,Unknown,International,None
3001,2,https://gtld-servers.net,gtld-servers.net,gtld-servers,net,net,simple,Unknown,International,None
3002,3,https://cloudflare.com,cloudflare.com,cloudflare,com,com,simple,Unknown,International,None
3003,4,https://gstatic.com,gstatic.com,gstatic,com,com,simple,Unknown,International,None
3004,5,https://facebook.com,facebook.com,facebook,com,com,simple,Unknown,International,None
3005,6,https://microsoft.com,microsoft.com,microsoft,com,com,simple,Unknown,International,None
3006,7,https://googleapis.com,googleapis.com,googleapis,com,com,simple,Unknown,International,None
3007,8,https://youtube.com,youtube.com,youtube,com,com,simple,Unknown,International,None
3008,9,https://amazonaws.com,amazonaws.com,amazonaws,com,com,simple,Unknown,International,None
3009,10,https://apple.com,apple.com,apple,com,com,simple,Unknown,International,None


### let's save the files
- tranco_clean.csv = full cleaned dataset (all European + International rows)
- urls.csv = your 5000-row working sample for the scraper


In [36]:
df_clean.to_csv('../clean_data/tranco_clean.csv', index=False)
print(f'Saved tranco_clean.csv — {len(df_clean)} rows')

df_work.to_csv('../clean_data/urls.csv', index=False)
print(f'Saved urls.csv — {len(df_work)} rows, ready for scraping')


Saved tranco_clean.csv — 691768 rows
Saved urls.csv — 5000 rows, ready for scraping


# Load and inspect the HTTPArchive data 
- (BigQuery -> flat file)
 
 we extracted  3 csv files from bigquery, Now we need to merge them using Pytheon , because it's much easier to understand and debug if needed

In [3]:
df_summary = pd.read_csv('../raw_data/bigquery_summary.csv')
df_tech    = pd.read_csv('../raw_data/bigquery_technologies.csv')
df_cmp     = pd.read_csv('../raw_data/bigquery_cmp_names.csv').drop_duplicates(subset='page')

In [38]:
# Merge using the correct variable names throughout
df = df_summary.merge(df_tech, on='page', how='left')
df = df.merge(df_cmp, on='page', how='left')

In [39]:
# Fill missing values now that everything is merged
df['has_analytics']   = df['has_analytics'].fillna(0)
df['has_advertising'] = df['has_advertising'].fillna(0)
df['has_cmp']         = df['has_cmp'].fillna(0)
df['tech_count']      = df['tech_count'].fillna(0)
df['cmp_name']        = df['cmp_name'].fillna('none')

In [40]:
print(df['has_analytics'].value_counts())

has_analytics
1.0    4008
0.0     992
Name: count, dtype: int64


In [41]:
df['has_advertising'].value_counts()

has_advertising
0.0    2546
1.0    2454
Name: count, dtype: int64

In [42]:
df.to_csv('../clean_data/bigquery_sample.csv', index=False)